# Customer Churn — Who, Why, and What to Do

**Business question:** *Which customers are about to leave, what's driving them away, and which interventions will move the needle?*

We don't stop at a churn score — we use **SHAP** to explain the drivers and translate them into retention actions. (Data is synthetic with known drivers, so we can verify the explanations are correct.)

In [ ]:
import sys; sys.path.append('..')
from src.run_pipeline import run
out = run()
print(f"Churn rate: {out['churn_rate']:.1%}")
out['results']

## Top churn drivers (global SHAP importance)

In [ ]:
import matplotlib.pyplot as plt
d = out['drivers'].head(10).iloc[::-1]
colors = ['#d6336c' if '+' in e else '#1c7ed6' for e in d['effect']]
plt.figure(figsize=(9,5)); plt.barh(d['feature'], d['importance'], color=colors)
plt.title('Top churn drivers (red = increases churn)'); plt.xlabel('mean |SHAP|'); plt.show()
out['drivers'].head(10)

## Recommended retention actions

In [ ]:
for i, r in enumerate(out['recommendations'], 1):
    print(f'{i}. {r}')

## Per-customer reason codes

For operations, each at-risk customer needs a *reason*, not just a number.

In [ ]:
from src.data import load_raw, split
from src.model import build_xgboost
from src.explain import explain_customer
ds = split(load_raw())
pipe = build_xgboost().fit(ds.X_train, ds.y_train)
explain_customer(pipe, ds.X_test.head(1))

## Takeaways

- **Top 3 drivers:** short tenure, month-to-month contracts, and high monthly charges.
- SHAP recovered the *true* (synthetic) drivers with correct direction — the explanations are trustworthy.
- The recommended actions (onboarding, annual-contract incentives, plan right-sizing, cross-sell, proactive support) target those drivers directly.
- **Business translation:** prioritize retention spend on month-to-month, high-bill, low-tenure customers — that's where the model says the risk and the leverage are.